# 01 - SQL Server → MinIO (Landing Zone)

Lê todas as tabelas do `LojaVirtualDB` e grava no bucket `landing-zone` do MinIO no formato CSV.

```
SQL Server (LojaVirtualDB)  →  MinIO / landing-zone / <tabela> / <tabela>.csv
```

> Execute o notebook `00_setup_sqlserver.ipynb` antes deste.


In [1]:
import pyodbc
import pandas as pd
import boto3
import os
from io import StringIO
from botocore.exceptions import ClientError
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

# SQL Server
DRIVER   = "{ODBC Driver 18 for SQL Server}"
SERVER   = f"{os.getenv('SQLSERVER_HOST')},{os.getenv('SQLSERVER_PORT')}"
DATABASE = os.getenv("SQLSERVER_DB")
USER     = os.getenv("SQLSERVER_USER")
PASSWORD = os.getenv("SQLSERVER_PASSWORD")

# MinIO
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS   = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET   = os.getenv("MINIO_SECRET_KEY")
LANDING_BUCKET = "landing-zone"

In [2]:
# Lê as tabelas do SQL Server para DataFrames
conn = pyodbc.connect(
    f"DRIVER={DRIVER};SERVER={SERVER};DATABASE={DATABASE};UID={USER};PWD={PASSWORD};TrustServerCertificate=yes;"
)

tabelas = ["clientes", "produtos", "pedidos"]
dados = {t: pd.read_sql(f"SELECT * FROM {t}", conn) for t in tabelas}
conn.close()

for nome, df in dados.items():
    print(f"{nome}: {len(df)} registros | colunas: {list(df.columns)}")

clientes: 5 registros | colunas: ['id', 'nome', 'email', 'cidade']
produtos: 5 registros | colunas: ['id', 'nome', 'categoria', 'preco', 'estoque']
pedidos: 5 registros | colunas: ['id', 'cliente_id', 'produto_id', 'quantidade', 'valor_total', 'data_pedido', 'status']


C:\Users\aliss_16u0t6a\AppData\Local\Temp\ipykernel_9044\4105923802.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dados = {t: pd.read_sql(f"SELECT * FROM {t}", conn) for t in tabelas}


In [ ]:
# Conecta ao MinIO e garante que o bucket landing-zone existe
s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS,
    aws_secret_access_key=MINIO_SECRET,
)

try:
    s3.create_bucket(Bucket=LANDING_BUCKET)
    print(f"Bucket '{LANDING_BUCKET}' criado.")
except ClientError as e:
    code = e.response["Error"]["Code"]
    if code in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        print(f"Bucket '{LANDING_BUCKET}' já existia.")
    else:
        raise

Bucket 'landing-zone' já existia.


In [4]:
# Converte os DataFrames para CSV e envia ao MinIO
for nome, df in dados.items():
    buffer = StringIO()
    df.to_csv(buffer, index=False)
    key = f"{nome}/{nome}.csv"
    s3.put_object(
        Bucket=LANDING_BUCKET,
        Key=key,
        Body=buffer.getvalue().encode("utf-8"),
    )
    print(f"Enviado → s3://{LANDING_BUCKET}/{key}  ({len(df)} linhas)")

Enviado → s3://landing-zone/clientes/clientes.csv  (5 linhas)
Enviado → s3://landing-zone/produtos/produtos.csv  (5 linhas)
Enviado → s3://landing-zone/pedidos/pedidos.csv  (5 linhas)


In [5]:
# Lista os objetos no bucket para confirmar o envio
resp = s3.list_objects_v2(Bucket=LANDING_BUCKET)
print(f"Conteúdo do bucket '{LANDING_BUCKET}':")
for obj in resp.get("Contents", []):
    print(f"  {obj['Key']}  ({obj['Size']} bytes)")

Conteúdo do bucket 'landing-zone':
  clientes/clientes.csv  (223 bytes)
  pedidos/pedidos.csv  (254 bytes)
  produtos/produtos.csv  (231 bytes)
